<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>GraphRAG</h1>
<h1>2. Text to Triples with spaCy, fastcoref, and REBEL</h1>
        <p>Bruno Gonçalves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In [ ]:
import hashlib
import json
from collections import Counter
from pathlib import Path

import spacy
from spacy.cli import download

import torch
from spacy import displacy
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

from thinc.api import get_current_ops

# Importing this module registers the "fastcoref" factory with spaCy —
# without it, add_pipe raises E002 "Can't find factory for 'fastcoref'".
from fastcoref import spacy_component  # noqa: F401
from fastcoref.modeling import FCorefModel

import matplotlib.pyplot as plt 
from tqdm.auto import tqdm

import watermark

%load_ext watermark
%matplotlib inline

We start by print out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.12.13
IPython version      : 9.15.0

Compiler    : Clang 22.1.3 
OS          : Darwin
Release     : 25.5.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: 44191938ab62c43da1cc2feb9fa73f73247587e9

json        : 2.0.9
matplotlib  : 3.11.0
spacy       : 3.8.14
torch       : 2.12.1
tqdm        : 4.68.3
transformers: 5.13.0
watermark   : 2.6.0



Load default figure style

In [3]:
plt.style.use('d4sci.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

And detect what hardware we're working with

In [ ]:
if torch.cuda.is_available():
    DEVICE = "cuda"          # NVIDIA GPU
elif torch.backends.mps.is_available():
    DEVICE = "mps"           # Apple Silicon GPU
else:
    DEVICE = "cpu"

print(f"transformer models (fastcoref, REBEL) will run on: {DEVICE}")

# spaCy's acceleration is SEPARATE from torch's: it runs on cupy for NVIDIA
# GPUs (install spacy[cudaXX]) and on Apple's AMX matrix coprocessor via
# thinc-apple-ops on Apple Silicon (install spacy[apple] — several times
# faster NER than plain numpy). prefer_gpu() is a no-op without cupy.
spacy.prefer_gpu()
print(f"spaCy compute backend: {get_current_ops().name}")

transformer models (fastcoref, REBEL) will run on: mps
spaCy compute backend: mps


In [5]:
# --- Checkpoint system ----------------------------------------------------------
# Every part of this workshop READS its inputs from, and WRITES its outputs to,
# a shared `checkpoints/` directory next to the notebooks. If an expected
# checkpoint is missing (you skipped a part, or a stage failed on your machine),
# we fall back to built-in data so THIS notebook still runs end to end.
# That mirrors good workshop practice: nobody gets stranded because an earlier
# stage broke — you load the checkpoint and keep moving.
CKPT = Path("checkpoints")
CKPT.mkdir(exist_ok=True)

def save_text(name, text):
    (CKPT / name).write_text(text, encoding="utf-8")
    print(f"[checkpoint] saved  {CKPT / name}  ({len(text):,} chars)")

def load_text(name, fallback=None):
    p = CKPT / name
    if p.exists():
        print(f"[checkpoint] loaded {p}")
        return p.read_text(encoding="utf-8")
    print(f"[checkpoint] {p} NOT FOUND — using built-in fallback data")
    return fallback

def save_jsonl(name, rows):
    with open(CKPT / name, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    print(f"[checkpoint] saved  {CKPT / name}  ({len(rows)} rows)")

def load_jsonl(name, fallback=None):
    p = CKPT / name
    if p.exists():
        rows = [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]
        print(f"[checkpoint] loaded {p}  ({len(rows)} rows)")
        return rows
    print(f"[checkpoint] {p} NOT FOUND — using built-in fallback data")
    return fallback

# --- Incremental checkpoints for long-running stages ------------------------------
# On the FULL wikitext-103 corpus (~526M chars), Stages 1-3 take hours (Stage 3:
# days on this hardware). So every heavy stage below streams its output to disk
# as it goes and records how far it got in a small `.progress.json` sidecar,
# keyed on a fingerprint of its input. The rules that fall out of this:
#   * interrupt the kernel any time — the next run RESUMES where it stopped
#   * a finished stage just loads its checkpoint in seconds (the "fast re-run")
#   * if the input text changes, the fingerprint changes and the stage restarts

def fingerprint(text):
    """Cheap identity for a (possibly huge) text — avoids hashing all 526M chars."""
    key = f"{len(text)}:{text[:1000]}:{text[-1000:]}"
    return hashlib.sha1(key.encode("utf-8")).hexdigest()[:12]

def stage_progress(name, fp):
    """Load a stage's saved state; discard it if the input fingerprint changed."""
    p = CKPT / name
    if p.exists():
        state = json.loads(p.read_text(encoding="utf-8"))
        if state.get("fp") == fp:
            return state
        print(f"[checkpoint] {p} is stale (input changed) — restarting stage")
    return {"fp": fp, "done": 0, "complete": False}

def save_stage_progress(name, state):
    (CKPT / name).write_text(json.dumps(state), encoding="utf-8")

def append_jsonl(name, rows, reset=False):
    with open(CKPT / name, "w" if reset else "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

def append_text(name, text, reset=False):
    with open(CKPT / name, "w" if reset else "a", encoding="utf-8") as f:
        f.write(text)

def chunk_text(text, max_chars):
    """Split text into chunks of at most max_chars, cutting at sentence-ish
    boundaries. (wikitext ships pre-tokenized, so sentence ends look like ' . ')"""
    chunks, start = [], 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        if end < len(text):
            for sep in (" . ", ". ", " "):
                cut = text.rfind(sep, start, end)
                if cut > start:
                    end = cut + len(sep)
                    break
        chunks.append(text[start:end])
        start = end
    return chunks

Read the  `corpus.txt` checkpoint

In [6]:
FALLBACK_TEXT = (
    "Marie Curie was a Polish and naturalised-French physicist and chemist. She conducted pioneering research on radioactivity. She was the first woman to win a Nobel Prize, and she remains the only person to win Nobel Prizes in two scientific fields. Marie Curie was born in Warsaw. She studied at the University of Paris, where she met Pierre Curie. Pierre Curie was a French physicist. He shared the 1903 Nobel Prize in Physics with Marie Curie and Henri Becquerel. Marie Curie married Pierre Curie in 1895. After Pierre died in 1906, Marie Curie took over his professorship at the University of Paris. She became the first woman to teach at the university. Marie Curie founded the Curie Institute in Paris in 1920. It remains a major centre for medical research today. Her daughter Irene Joliot-Curie was also a chemist. Irene Joliot-Curie won the Nobel Prize in Chemistry in 1935 with her husband Frederic Joliot-Curie."
)

raw_text = load_text("corpus.txt", fallback=FALLBACK_TEXT)

CORPUS_FP = fingerprint(raw_text)
print(f"\nCorpus: {len(raw_text):,} characters · fingerprint {CORPUS_FP}")

[checkpoint] loaded checkpoints/corpus.txt

Corpus: 526,514,281 characters · fingerprint aa8880e24ff0


---
## Stage 1 — spaCy: sentences and named entities

spaCy gives us two things:

1. **Sentence segmentation.** REBEL (Stage 3) works best on sentence-sized inputs, so we need reliable boundaries — harder than splitting on periods, thanks to abbreviations like *"Dr."* and *"1903."*
2. **Named-entity recognition (NER).** A statistical model tags spans as `PERSON`, `ORG`, `GPE` (geopolitical entity), `DATE`, etc. We use these to *inspect* what the text contains — and entity labels also help filter noisy triples later.

We use `en_core_web_sm` — small, fast, CPU-friendly; slightly less accurate than the larger `md`/`lg`/`trf` models, a fine workshop trade-off.

**Scaling to the full corpus.** Two changes versus the naive `nlp(raw_text)` call:

1. **Stream in chunks.** spaCy caps single documents at 1M characters (E088) — and needs ~1GB of memory per 100k characters with heavy components. We split the corpus at sentence boundaries into ~100k-char chunks and stream them through `nlp.pipe`.
2. **Swap the parser for `senter`.** We only need sentence boundaries and NER; the dependency parser would multiply the runtime several-fold for annotations we never use.

Entity rows stream straight to `entities.jsonl` as each chunk finishes, and a progress sidecar tracks how far we got — interrupt any time and re-run to resume; once complete, this cell loads from the checkpoint in seconds. With hardware-accelerated ops (see the device cell) the first full pass measures **~450k chars/s ≈ 20 minutes** on this corpus; several times slower on plain numpy.

In [ ]:
# Load the small English pipeline with ONLY the pieces this stage needs:
# tok2vec + ner for entities, senter for fast sentence boundaries.
SPACY_EXCLUDE = ["parser", "tagger", "attribute_ruler", "lemmatizer"]
try:
    nlp = spacy.load("en_core_web_sm", exclude=SPACY_EXCLUDE)
except OSError:                                   # model not downloaded yet
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm", exclude=SPACY_EXCLUDE)
nlp.enable_pipe("senter")            # shipped disabled; replaces the parser
print("pipeline:", nlp.pipe_names)

# Split the corpus at sentence boundaries into pipeline-sized chunks, and
# precompute each chunk's character offset so entity positions stay global.
NER_CHUNK_CHARS = 100_000
chunks = chunk_text(raw_text, NER_CHUNK_CHARS)
offsets = [0]
for c in chunks:
    offsets.append(offsets[-1] + len(c))
print(f"{len(chunks):,} chunks of ≤{NER_CHUNK_CHARS:,} chars")

# --- Resumable streaming pass (CHECKPOINT OUT: entities.jsonl) --------------------
# Entity rows are appended to entities.jsonl as each chunk completes; running
# label counts, token/sentence totals and a small display sample live in the
# progress sidecar so a resumed run doesn't lose them.
ner_state = stage_progress("entities.progress.json", CORPUS_FP)

if not ner_state["complete"]:
    if ner_state["done"] == 0:
        append_jsonl("entities.jsonl", [], reset=True)      # fresh file
    start = ner_state["done"]
    labels = Counter(ner_state.get("labels", {}))
    sample = ner_state.get("sample", [])
    for i, chunk_doc in enumerate(
            tqdm(nlp.pipe(chunks[start:], batch_size=8),
                 total=len(chunks), initial=start, desc="spaCy senter+NER"),
            start=start):
        base = offsets[i]
        rows = [{"text": e.text, "label": e.label_,
                 "start": base + e.start_char, "end": base + e.end_char}
                for e in chunk_doc.ents]
        append_jsonl("entities.jsonl", rows)
        labels.update(r["label"] for r in rows)
        for r in rows:
            if len(sample) >= 15:
                break
            if r["text"] not in {s["text"] for s in sample}:
                sample.append(r)
        ner_state.update(done=i + 1, labels=dict(labels), sample=sample,
                         n_ents=ner_state.get("n_ents", 0) + len(rows),
                         n_tokens=ner_state.get("n_tokens", 0) + len(chunk_doc),
                         n_sents=ner_state.get("n_sents", 0) + sum(1 for _ in chunk_doc.sents))
        save_stage_progress("entities.progress.json", ner_state)
    ner_state["complete"] = True
    save_stage_progress("entities.progress.json", ner_state)
else:
    print("[checkpoint] Stage 1 already complete — loaded stats from sidecar")

print(f"\n{ner_state['n_sents']:,} sentences · {ner_state['n_tokens']:,} tokens "
      f"· {ner_state['n_ents']:,} entity mentions")

# A small preview Doc of the corpus opening, for the inspection cells below.
preview_doc = nlp(chunks[0])
sentences = list(preview_doc.sents)
print("\nFirst 3 sentences:")
for s in sentences[:3]:
    print(" •", s.text.strip())

pipeline: ['tok2vec', 'senter', 'ner']
5,271 chunks of ≤100,000 chars


spaCy senter+NER:  34%|###4      | 1816/5271 [00:00<?, ?it/s]

In [ ]:
# --- Inspect the named entities ----------------------------------------------
# Counting entities by label gives a quick fingerprint of WHAT the text is
# about before we transform anything. The counts were accumulated during the
# streaming pass above (they cover the FULL corpus, not just a sample).
label_counts = Counter(ner_state["labels"])
print("Entity types found:")
for label, n in label_counts.most_common():
    print(f"  {n:>9,} × {label}")

print("\nSample entities (from the corpus opening):")
for ent in ner_state["sample"]:
    print(f"  {ent['label']:<12} {ent['text']}")

Entity types found: {'ORG': 8, 'PERSON': 7, 'GPE': 5, 'WORK_OF_ART': 5, 'CARDINAL': 4, 'NORP': 4, 'DATE': 4, 'ORDINAL': 3, 'LAW': 3, 'EVENT': 2, 'PRODUCT': 1}

Sample entities:
  PERSON   Senjō no
  CARDINAL 3
  NORP     Japanese
  PERSON   Valkyria
  GPE      the Battlefield 3
  PERSON   Valkyria Chronicles III
  GPE      Japan
  PERSON   Sega
  ORG      Media
  DATE     January 2011
  ORDINAL  third
  ORDINAL  first
  WORK_OF_ART Nameless
  EVENT    the Second Europan War
  ORG      Imperial


In [ ]:
# displacy renders entities inline — 30 seconds of gawking that builds real
# intuition for what the model does (and doesn't) catch.
first_sents = " ".join(s.text for s in sentences[:4])
displacy.render(nlp(first_sents), style="ent", jupyter=True)

In [ ]:
# --- CHECKPOINT OUT: entities.jsonl ---------------------------------------------
# One row per entity mention, with label and GLOBAL character offsets — already
# written incrementally during the streaming pass above. Not consumed by later
# parts of THIS pipeline, but exactly what you'd feed entity-level analytics
# or a type-validation pass on triples (see Stage 3 notes).
p = CKPT / "entities.jsonl"
print(f"[checkpoint] {p}  ({p.stat().st_size / 1e6:,.1f} MB · "
      f"{ner_state['n_ents']:,} rows)")

[checkpoint] saved  checkpoints/entities.jsonl  (46 rows)


**What to notice before moving on:** NER finds entity *mentions*, but has no idea that *"Marie Curie"*, *"Curie"*, and *"She"* are the same person. Pronouns aren't entities at all — every fact expressed through one is currently invisible. That's Stage 2's job.

---
## Stage 2 — fastcoref: resolving who "she" is

**Coreference resolution** groups all mentions that refer to the same real-world entity — *"Marie Curie"*, *"She"*, *"her"* — into clusters, and lets us rewrite the text with pronouns replaced by their canonical mention.

**Why this makes or breaks the graph.** Triple extraction is sentence-local: REBEL sees one sentence at a time and cannot know the *"She"* in sentence 5 is the *"Marie Curie"* of sentence 1. Without resolution:

```
BROKEN:  ("She", "award received", "Nobel Prize")     ← orphan node "She"
FIXED:   ("Marie Curie", "award received", "Nobel Prize")
```

Every unresolved pronoun either creates a junk node or silently drops a fact. This is the single highest-leverage line of code in the pipeline.

**fastcoref** plugs into spaCy as a pipeline component; with `resolve_text=True` it returns the rewritten text directly. First use downloads the model weights. **Robustness note:** if fastcoref fails on your machine, the `except` branch below substitutes a pre-resolved fallback (when you're on the fallback corpus) or the raw text (with a loud warning) — the pipeline continues either way, and you'll *see* the damage skipping coref causes in your graph.

In [ ]:
# --- Build a second spaCy pipeline with the coref component ----------------------
# Kept separate from `nlp` so Stage 1 stays reproducible on its own.
try:
    # Compatibility shim: fastcoref predates transformers 5.x, which expects
    # every model class to expose `all_tied_weights_keys`. Without this,
    # loading raises AttributeError and coref silently falls back — the exact
    # "quietly broken graph" failure this stage exists to prevent.
    if not hasattr(FCorefModel, "all_tied_weights_keys"):
        FCorefModel.all_tied_weights_keys = {}

    # fastcoref brings its own transformer, but its text-rewriting logic reads
    # token.pos_/token.tag_ from the host pipeline to pick cluster heads and
    # handle possessives — so the tagger MUST stay. Parser and NER can go.
    nlp_coref = spacy.load("en_core_web_sm",
                           exclude=["parser", "ner", "lemmatizer"])
    # component registered by the fastcoref package; run it on the GPU when we
    # have one (its per-call progress bars are off — we have our own).
    # max_tokens_in_batch=25k keeps the GPU fed: ~1.4x faster than the 10k
    # default on Apple MPS in our benchmark (50k was no better).
    nlp_coref.add_pipe("fastcoref", config={"device": DEVICE,
                                            "max_tokens_in_batch": 25_000,
                                            "enable_progress_bar": False})
    print("coref pipeline:", nlp_coref.pipe_names)
    COREF_OK = True
except Exception as e:
    print(f"fastcoref unavailable ({type(e).__name__}: {e}) — will use fallback.")
    COREF_OK = False

07/05/2026 17:33:27 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07/05/2026 17:33:27 - WARNING - 	 Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
07/05/2026 17:33:27 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/f-coref/e91dfff12879495d882ba9460d0c5d5dd44ade59/config.json "HTTP/1.1 200 OK"
07/05/2026 17:33:27 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07/05/2026 17:33:27 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/f-coref/e91dfff12879495d882ba9460d0c5d5dd44ade59/config.json "HTTP/1.1 200 OK"
07/05/2026 17:33:27 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
07/0

Loading weights:   0%|          | 0/133 [00:00<?, ?it/s]

07/05/2026 17:33:27 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/commits/main "HTTP/1.1 200 OK"
[transformers] FCorefModel LOAD REPORT from: biu-nlp/f-coref
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


fastcoref unavailable (AttributeError: 'FCorefModel' object has no attribute 'all_tied_weights_keys') — will use fallback.


In [ ]:
# --- Before/after on a tiny example ----------------------------------------------
# A sentence pair engineered to break naive pipelines:
demo = ("Marie Curie studied at the University of Paris. "
        "She met Pierre Curie there, and he shared her passion for science.")

if COREF_OK:
    demo_doc = nlp_coref(demo, component_cfg={"fastcoref": {"resolve_text": True}})
    print("BEFORE:", demo)
    print("\nClusters found:", demo_doc._.coref_clusters)
    print("\nAFTER: ", demo_doc._.resolved_text)
    # Expect: She→Marie Curie, he→Pierre Curie, her→Marie Curie('s).
    # Three pronouns, three facts rescued.
else:
    print("(fastcoref unavailable — skipping live demo)")

(fastcoref unavailable — skipping live demo)


In [ ]:
# --- Resolve the whole corpus (with graceful fallback) -----------------------------
FALLBACK_RESOLVED = (
    "Marie Curie was a Polish and naturalised-French physicist and chemist. Marie Curie conducted pioneering research on radioactivity. Marie Curie was the first woman to win a Nobel Prize, and Marie Curie remains the only person to win Nobel Prizes in two scientific fields. Marie Curie was born in Warsaw. Marie Curie studied at the University of Paris, where Marie Curie met Pierre Curie. Pierre Curie was a French physicist. Pierre Curie shared the 1903 Nobel Prize in Physics with Marie Curie and Henri Becquerel. Marie Curie married Pierre Curie in 1895. After Pierre Curie died in 1906, Marie Curie took over Pierre Curie's professorship at the University of Paris. Marie Curie became the first woman to teach at the University of Paris. Marie Curie founded the Curie Institute in Paris in 1920. The Curie Institute remains a major centre for medical research today. Marie Curie's daughter Irene Joliot-Curie was also a chemist. Irene Joliot-Curie won the Nobel Prize in Chemistry in 1935 with Irene Joliot-Curie's husband Frederic Joliot-Curie."
)

# Coref runs on chapter-sized chunks: model quality drops on very long inputs
# (see the caveats below), and chunking is what lets us stream 526M characters
# through a transformer at all. Resolved text is APPENDED to resolved.txt as
# each chunk finishes; the sidecar makes the pass resumable, and once complete
# this cell just reads the file back.
COREF_CHUNK_CHARS = 10_000

if COREF_OK:
    coref_chunks = chunk_text(raw_text, COREF_CHUNK_CHARS)
    coref_state = stage_progress("resolved.progress.json", CORPUS_FP)
    if not coref_state["complete"]:
        if coref_state["done"] == 0:
            append_text("resolved.txt", "", reset=True)     # fresh file
        start = coref_state["done"]
        # Batched streaming: the component's pipe() groups chunks into GPU
        # batches, but still yields one resolved doc at a time — so we keep
        # per-chunk checkpoint granularity at batch-level throughput.
        stream = nlp_coref.pipe(
            coref_chunks[start:], batch_size=32,
            component_cfg={"fastcoref": {"resolve_text": True, "batch_size": 32}})
        for i, chunk_doc in enumerate(
                tqdm(stream, total=len(coref_chunks), initial=start,
                     desc="fastcoref"), start=start):
            append_text("resolved.txt", chunk_doc._.resolved_text + " ")
            coref_state["done"] = i + 1
            coref_state["n_clusters"] = (coref_state.get("n_clusters", 0)
                                         + len(chunk_doc._.coref_clusters))
            save_stage_progress("resolved.progress.json", coref_state)
        coref_state["complete"] = True
        save_stage_progress("resolved.progress.json", coref_state)
    else:
        print("[checkpoint] Stage 2 already complete — reading resolved.txt")
    resolved_text = (CKPT / "resolved.txt").read_text(encoding="utf-8")
    print(f"{coref_state['n_clusters']:,} coreference clusters resolved")
elif raw_text == FALLBACK_TEXT:
    # We know this text — use its hand-resolved version.
    resolved_text = FALLBACK_RESOLVED
    print("Using pre-resolved fallback text.")
else:
    resolved_text = raw_text
    print("WARNING: proceeding WITHOUT coreference resolution. Expect orphaned")
    print("pronoun nodes in your graph — the exact failure this stage prevents.")

print("\nFirst 400 characters of resolved text:\n" + resolved_text[:400] + " …")

pronoun nodes in your graph — the exact failure this stage prevents.

First 400 characters of resolved text:
Senjō no Valkyria 3 : <unk> Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical an …


In [ ]:
# --- CHECKPOINT OUT: resolved.txt --------------------------------------------------
# When coref ran above, resolved.txt was streamed to disk chunk by chunk.
# In the fallback branches we still need to persist it here.
if COREF_OK:
    print(f"[checkpoint] {CKPT / 'resolved.txt'} already written incrementally "
          f"({len(resolved_text):,} chars)")
else:
    save_text("resolved.txt", resolved_text)

[checkpoint] saved  checkpoints/resolved.txt  (4,623 chars)


**Caveats worth knowing** (intellectual honesty beats a tidy demo):

- Coref models make mistakes, and a *wrong* resolution is worse than none — it creates a confidently false fact. Spot-check a sample.
- Substituting long canonical mentions for every pronoun makes text stilted and occasionally garbles grammar. REBEL doesn't care about elegance; acceptable cost here.
- Resolution quality drops on very long documents; processing chapter-sized chunks is standard practice.

---
## Stage 3 — REBEL: from sentences to (head, relation, tail) triples

**REBEL** (Relation Extraction By End-to-end Language generation) is a seq2seq transformer (BART fine-tuned on Wikipedia/Wikidata) that reads a sentence and *generates* its triples as a string, using special tokens as delimiters:

```
<triplet> Marie Curie <subj> Nobel Prize <obj> award received
```

- `<triplet>` starts a new triple and introduces the **head** entity
- `<subj>` introduces the **tail** entity
- `<obj>` introduces the **relation** label

(Confusingly named tokens — they're delimiters, not grammatical roles. The parser below is the standard decoding recipe from the model's authors.)

Two advantages of this approach: relations come from a **fixed vocabulary** of ~200 Wikidata-style relation types (`spouse`, `place of birth`, `award received`, …), so the graph's edge types stay consistent — and it's end-to-end, no hand-written extraction rules.

First run downloads ~1.6 GB of weights. If the download fails, the `except` branch substitutes a built-in triple set so Parts 3–4 still have input.

**This is by far the longest stage on the full corpus.** Extraction costs ~1s per sentence on CPU; on GPU with fp16 weights and batched generation we measured **~35 ms/sentence on Apple MPS** — still ~1.5–2 days across the ~4M sentences of wikitext-103. The extraction loop below is built for exactly that reality: triples are appended to disk batch by batch and progress is checkpointed, so you can interrupt the kernel whenever you like, run it overnight in installments, and resume without losing anything. Once the pass is complete, re-runs load the finished checkpoint in seconds. Part 3 can also be pointed at a *partial* `triples.jsonl` if you don't want to wait for full coverage.

In [ ]:
# --- Load REBEL ----------------------------------------------------------------
# We load tokenizer+model directly (not a pipeline) so we can KEEP the special
# delimiter tokens in the decoded output — the parser needs them.
REBEL = "Babelscape/rebel-large"
# On GPU (cuda/mps) we load the weights in float16: half the memory, ~10-15%
# faster generation, and identical triples on our spot checks. CPU stays in
# float32 — fp16 is slower there, not faster.
REBEL_DTYPE = torch.float16 if DEVICE != "cpu" else torch.float32
try:
    rebel_tokenizer = AutoTokenizer.from_pretrained(REBEL)
    rebel_model = AutoModelForSeq2SeqLM.from_pretrained(REBEL, dtype=REBEL_DTYPE).to(DEVICE)
    rebel_model.eval()
    REBEL_OK = True
    print(f"REBEL loaded on {DEVICE} ({REBEL_DTYPE})")
except Exception as e:
    REBEL_OK = False
    print(f"REBEL unavailable ({type(e).__name__}) — will use fallback triples.")

07/05/2026 17:33:27 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/discussions?p=0 "HTTP/1.1 200 OK"
07/05/2026 17:33:27 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/commits/refs%2Fpr%2F1 "HTTP/1.1 200 OK"
07/05/2026 17:33:27 - INFO - 	 HTTP Request: HEAD https://huggingface.co/Babelscape/rebel-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07/05/2026 17:33:27 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Babelscape/rebel-large/44eb6cb4585df284ce6c4d6a7013f83fe473c052/config.json "HTTP/1.1 200 OK"
07/05/2026 17:33:27 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/refs%2Fpr%2F1/model.safetensors.index.json "HTTP/1.1 404 Not Found"
07/05/2026 17:33:27 - INFO - 	 HTTP Request: HEAD https://huggingface.co/Babelscape/rebel-large/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
07/05/2026 17:33:27 - INFO - 	 HTTP Request: HEAD h

model.safetensors:   0%|          | 0.00/362M [00:00<?, ?B/s]

07/05/2026 17:33:28 - INFO - 	 HTTP Request: HEAD https://huggingface.co/Babelscape/rebel-large/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
07/05/2026 17:33:28 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Babelscape/rebel-large/44eb6cb4585df284ce6c4d6a7013f83fe473c052/tokenizer_config.json "HTTP/1.1 200 OK"
07/05/2026 17:33:28 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/Babelscape/rebel-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
07/05/2026 17:33:28 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/Babelscape/rebel-large/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
07/05/2026 17:33:28 - INFO - 	 HTTP Request: HEAD https://huggingface.co/Babelscape/rebel-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07/05/2026 17:33:28 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Babelscape/rebel-large

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

07/05/2026 17:33:28 - INFO - 	 HTTP Request: HEAD https://huggingface.co/Babelscape/rebel-large/resolve/main/generation_config.json "HTTP/1.1 404 Not Found"
07/05/2026 17:33:28 - INFO - 	 HTTP Request: HEAD https://huggingface.co/Babelscape/rebel-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
07/05/2026 17:33:28 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Babelscape/rebel-large/44eb6cb4585df284ce6c4d6a7013f83fe473c052/config.json "HTTP/1.1 200 OK"


REBEL loaded on mps


In [ ]:
# --- The standard REBEL output parser -------------------------------------------
# Walks the generated string token by token, switching state at each delimiter.
# Adapted from the model card's reference implementation.
def parse_rebel_output(text):
    triples = []
    head = relation = tail = ""
    current = None
    text = text.replace("<s>", "").replace("</s>", "").replace("<pad>", "").strip()
    for token in text.split():
        if token == "<triplet>":                  # new triple begins
            if head and relation and tail:
                triples.append((head.strip(), relation.strip(), tail.strip()))
            head, relation, tail = "", "", ""
            current = "head"
        elif token == "<subj>":                   # tail entity follows
            current = "tail"
        elif token == "<obj>":                    # relation label follows
            current = "relation"
        elif current == "head":
            head += " " + token
        elif current == "tail":
            tail += " " + token
        elif current == "relation":
            relation += " " + token
    if head and relation and tail:                # flush the final triple
        triples.append((head.strip(), relation.strip(), tail.strip()))
    return triples

def extract_triples_batch(sents):
    """Run REBEL on a BATCH of sentences → one triple-list per sentence.
    Batching amortizes generation overhead — the difference between weeks
    and days on the full corpus."""
    inputs = rebel_tokenizer(sents, max_length=256, truncation=True,
                             padding=True, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = rebel_model.generate(**inputs, max_length=256,
                                   num_beams=3,        # beams beat greedy here
                                   length_penalty=0.0)
    # Keep special tokens — the parser needs the delimiters:
    decoded = rebel_tokenizer.batch_decode(out, skip_special_tokens=False)
    return [parse_rebel_output(d) for d in decoded]

def extract_triples(sentence):
    """Run REBEL on one sentence → list of (head, relation, tail)."""
    return extract_triples_batch([sentence])[0]

if REBEL_OK:   # sanity check on one sentence
    print(extract_triples(
        "Marie Curie married Pierre Curie in 1895 in Sceaux, France."))

[('Marie Curie', 'spouse', 'Pierre Curie'), ('Pierre Curie', 'spouse', 'Marie Curie')]


In [ ]:
# --- Run extraction over the resolved corpus ---------------------------------------
# Sentence-by-sentence: REBEL was trained on short inputs, and per-sentence calls
# make failures easy to localize.
FALLBACK_TRIPLES = [
    {"head": "Marie Curie",          "relation": "country of citizenship", "tail": "Poland"},
    {"head": "Marie Curie",          "relation": "country of citizenship", "tail": "France"},
    {"head": "Marie Curie",          "relation": "occupation",             "tail": "physicist"},
    {"head": "Marie Curie",          "relation": "occupation",             "tail": "chemist"},
    {"head": "Marie Curie",          "relation": "field of work",          "tail": "radioactivity"},
    {"head": "Marie Curie",          "relation": "place of birth",         "tail": "Warsaw"},
    {"head": "Marie Curie",          "relation": "educated at",            "tail": "University of Paris"},
    {"head": "Marie Curie",          "relation": "employer",               "tail": "University of Paris"},
    {"head": "Marie Curie",          "relation": "award received",         "tail": "Nobel Prize in Physics"},
    {"head": "Pierre Curie",         "relation": "occupation",             "tail": "physicist"},
    {"head": "Pierre Curie",         "relation": "country of citizenship", "tail": "France"},
    {"head": "Pierre Curie",         "relation": "award received",         "tail": "Nobel Prize in Physics"},
    {"head": "Henri Becquerel",      "relation": "award received",         "tail": "Nobel Prize in Physics"},
    {"head": "Marie Curie",          "relation": "spouse",                 "tail": "Pierre Curie"},
    {"head": "Pierre Curie",         "relation": "spouse",                 "tail": "Marie Curie"},
    {"head": "Curie Institute",      "relation": "founded by",             "tail": "Marie Curie"},
    {"head": "Curie Institute",      "relation": "located in",             "tail": "Paris"},
    {"head": "Curie Institute",      "relation": "inception",              "tail": "1920"},
    {"head": "Marie Curie",          "relation": "child",                  "tail": "Irene Joliot-Curie"},
    {"head": "Irene Joliot-Curie",   "relation": "occupation",             "tail": "chemist"},
    {"head": "Irene Joliot-Curie",   "relation": "award received",         "tail": "Nobel Prize in Chemistry"},
    {"head": "Irene Joliot-Curie",   "relation": "spouse",                 "tail": "Frederic Joliot-Curie"},
    {"head": "Frederic Joliot-Curie","relation": "spouse",                 "tail": "Irene Joliot-Curie"},
    {"head": "Frederic Joliot-Curie","relation": "award received",         "tail": "Nobel Prize in Chemistry"},
    {"head": "Pierre Curie",         "relation": "date of death",          "tail": "1906"},
]

# --- Re-segment the RESOLVED text (chunked, cached) --------------------------------
# REBEL's resume logic below indexes into the sentence list, so the list must
# be IDENTICAL across runs — we therefore segment once and cache one sentence
# per line in resolved_sentences.txt. NER is switched off for this pass; we
# only need boundaries.
RESOLVED_FP = fingerprint(resolved_text)
SENT_FILE = CKPT / "resolved_sentences.txt"

sent_state = stage_progress("sentences.progress.json", RESOLVED_FP)
if not sent_state["complete"]:
    resolved_chunks = chunk_text(resolved_text, 100_000)
    with open(SENT_FILE, "w", encoding="utf-8") as f, \
         nlp.select_pipes(disable=["ner"]):
        for chunk_doc in tqdm(nlp.pipe(resolved_chunks, batch_size=8),
                              total=len(resolved_chunks), desc="sentence split"):
            for s in chunk_doc.sents:
                sent = " ".join(s.text.split())      # normalize whitespace
                if len(sent) > 10:
                    f.write(sent + "\n")
    sent_state["complete"] = True
    save_stage_progress("sentences.progress.json", sent_state)
else:
    print("[checkpoint] sentence segmentation already cached")

resolved_sents = SENT_FILE.read_text(encoding="utf-8").splitlines()
print(f"{len(resolved_sents):,} sentences to extract from")

# --- Resumable batched extraction (CHECKPOINT OUT: triples_raw.jsonl) --------------
# Raw triples are appended to triples_raw.jsonl one batch at a time and the
# sidecar records how many sentences are done. Interrupt the kernel whenever
# you like — the next run picks up at the same sentence. (The CLEANED triples
# that Parts 3-4 consume are saved separately as triples.jsonl below.)
REBEL_BATCH = 32          # sweet spot on Apple MPS (~35 ms/sentence measured);
                          # lower if you hit OOM, retune per GPU

if REBEL_OK:
    rebel_state = stage_progress("triples_raw.progress.json", RESOLVED_FP)
    if not rebel_state["complete"]:
        if rebel_state["done"] == 0:
            append_jsonl("triples_raw.jsonl", [], reset=True)     # fresh file
        done = rebel_state["done"]
        print(f"{len(resolved_sents) - done:,} sentences remaining — safe to "
              f"interrupt, progress is checkpointed every batch")
        pbar = tqdm(total=len(resolved_sents), initial=done, desc="REBEL")
        while done < len(resolved_sents):
            batch = resolved_sents[done:done + REBEL_BATCH]
            rows = [{"head": h, "relation": r, "tail": t}
                    for triple_list in extract_triples_batch(batch)
                    for h, r, t in triple_list]
            append_jsonl("triples_raw.jsonl", rows)
            done += len(batch)
            rebel_state.update(done=done,
                               n_triples=rebel_state.get("n_triples", 0) + len(rows))
            save_stage_progress("triples_raw.progress.json", rebel_state)
            pbar.update(len(batch))
        pbar.close()
        rebel_state["complete"] = True
        save_stage_progress("triples_raw.progress.json", rebel_state)
    else:
        print("[checkpoint] Stage 3 already complete — loading raw triples")
    raw_rows = load_jsonl("triples_raw.jsonl")
    all_triples = [(r["head"], r["relation"], r["tail"]) for r in raw_rows]
    print(f"\nExtracted {len(all_triples):,} raw triples from "
          f"{len(resolved_sents):,} sentences")
else:
    all_triples = [(t["head"], t["relation"], t["tail"]) for t in FALLBACK_TRIPLES]
    print(f"Using {len(all_triples)} built-in fallback triples.")

for t in all_triples[:12]:
    print("  ", t)

  sentence 1/33 · 1 triples so far
  sentence 11/33 · 11 triples so far
  sentence 21/33 · 22 triples so far
  sentence 31/33 · 31 triples so far

Extracted 34 raw triples from 33 sentences
   ('Senjō no Valkyria 3', 'language of work or name', 'Japanese')
   ('Valkyria Chronicles III', 'genre instance of developer', 'tactical role @-@ video game Sega')
   ('PlayStation Portable', 'part of the series', 'PlayStation')
   ('Valkyria series', 'country of origin', 'Japan')
   ('Second Europan War', 'participant', 'Imperial')
   ('Valkyria Chronicles II', 'publication date', '2010')
   ('standard features', 'part of', 'series')
   ('Valkyria Chronicles II', 'director', 'Takeshi Ozawa')
   ('script', 'practiced by', 'writers')
   ('opening theme', 'performer', "May 'n")
   ('Japanese', 'country', 'Japan')
   ('downloadable content', 'publication date', 'November of that year')


### Cleaning the triples

Raw extraction output is **noisy** — every information-extraction system's output is. Typical junk:

- **Duplicates** — the same fact stated (or beam-searched) twice.
- **Self-loops** — `(X, relation, X)`, usually a parsing artifact.
- **Degenerate entities** — empty strings, lone punctuation, absurdly long spans.

We apply a conservative filter. In production you'd go further: score triples with generation probabilities (a confidence proxy), validate entity types against the relation's expected signature (a `spouse` edge should join two `PERSON`s — this is where Stage 1's `entities.jsonl` earns its keep), and keep provenance (which sentence produced each triple) for auditability.

In [ ]:
def clean_triples(triples):
    seen, cleaned = set(), []
    for head, rel, tail in triples:
        h, r, t = head.strip(), rel.strip(), tail.strip()
        if not h or not r or not t:           # degenerate
            continue
        if h.lower() == t.lower():            # self-loop
            continue
        if len(h) > 80 or len(t) > 80:        # runaway span
            continue
        key = (h.lower(), r.lower(), t.lower())
        if key in seen:                       # duplicate
            continue
        seen.add(key)
        cleaned.append((h, r, t))
    return cleaned

triples = clean_triples(all_triples)
print(f"{len(all_triples)} raw → {len(triples)} clean triples\n")

print("Most common relations:")
for rel, n in Counter(r for _, r, _ in triples).most_common(10):
    print(f"  {n:>3} × {rel}")

34 raw → 34 clean triples

Most common relations:
   10 × subclass of
    3 × part of
    3 × instance of
    2 × publication date
    2 × facet of
    1 × language of work or name
    1 × genre instance of developer
    1 × part of the series
    1 × country of origin
    1 × participant


In [ ]:
# --- CHECKPOINT OUT: triples.jsonl --------------------------------------------------
# THE canonical artifact of the whole offline pipeline. JSONL of triples can
# rebuild the graph in ANY store — NetworkX today, Neo4j next year.
save_jsonl("triples.jsonl",
           [{"head": h, "relation": r, "tail": t} for h, r, t in triples])

[checkpoint] saved  checkpoints/triples.jsonl  (34 rows)


<center>
     <img src="data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>